# Activity C: Grounded Live-Corpus
### ISA Tutorial — CHIIR 2026

---

**Author:** Preetam Dammu, PhD Candidate, University of Washington · preetams@uw.edu  
**Please cite:** [Dammu & Roosta, CHIIR 2026](https://dl.acm.org/doi/abs/10.1145/3786304.3787893)

---

## Objective

Some questions **cannot** be answered from a fixed document set or a structured API because the answer changes day-to-day. This notebook shows the problem and the fix.

| Level | Name | Source |
|-------|------|--------|
| 2 | Structured Lookup | Deterministic API |
| 3 | Grounded Closed-Corpus | Fixed document set (RAG) |
| **4** | **Grounded Live-Corpus** | **Open web — live search** |

**What you will see:**
1. Ask the LLM a recent question → it fails (training cutoff)
2. Add DuckDuckGo live search → current, cited answer
3. Repeat for a few more questions to build intuition

> **Runtime:** Default model is `Qwen/Qwen2-0.5B-Instruct` (0.5 B params, CPU-friendly).  
> On Colab T4, swap to `microsoft/Phi-3.5-mini-instruct` for richer outputs.

In [1]:
# ── Install dependencies ────────────────────────────────────────────────────
# transformers       : HuggingFace model loading & text generation
# torch              : tensor operations (CPU or GPU)
# ddgs               : DuckDuckGo search library (free, keyless, no signup)

!pip install -q transformers torch ddgs

## Setting up the LLM

Same setup as Activity B. We load **Qwen2-0.5B-Instruct** once and reuse a single `generate(prompt)` helper throughout.

> **Want a stronger model?** On a Colab T4 GPU, swap `MODEL_ID` to  
> `"microsoft/Phi-3.5-mini-instruct"` (3.8 B params). The core lesson is identical.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Model selection ──────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"          # 0.5 B params, ~1 GB, works on CPU
# GPU upgrade (Colab T4 or better):
# MODEL_ID = "microsoft/Phi-3.5-mini-instruct"  # 3.8 B params, needs ~4 GB VRAM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"Loading {MODEL_ID} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
)
model = model.to(device)
model.eval()
print("Model ready!")


# ── Helper: generate ────────────────────────────────────────────────────────
def generate(prompt: str, max_new_tokens: int = 300) -> str:
    """
    Send a plain-text prompt to the LLM and return the response string.
    Uses the model's chat template for instruction-following format.
    Greedy decoding (do_sample=False) gives deterministic output.
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

/Users/preetams/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu
Loading Qwen/Qwen2-0.5B-Instruct ...
Model ready!


---

# Part 1 — The LLM Alone Fails on Live Questions

The LLM has a **training cutoff**. For anything after that date it can only hedge, give stale information, or hallucinate. Below we ask about an event that happened in early 2026 — well after the cutoff.

In [3]:
# ── 1.1  LLM-only attempt: no search context ────────────────────────────────
# Super Bowl LXI was played in February 2026 — the model's training
# cutoff predates this, so it cannot know the winner.

question = "Who won Super Bowl LXI in 2026?"

print("QUESTION:", question)
print("\n" + "─" * 60)
print("LLM answer (no search context):")
print("─" * 60)
llm_only_answer = generate(question)
print(llm_only_answer)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION: Who won Super Bowl LXI in 2026?

────────────────────────────────────────────────────────────
LLM answer (no search context):
────────────────────────────────────────────────────────────
As an AI language model, I cannot provide information about the 2026 Super Bowl or any other event that may have occurred before my training data was last updated. It is possible that there has been a change in the date of the Super Bowl or that the questioner may not be referring to a specific event. Please let me know if you need help with something else.


### What happened?

The model either hedged ("I don't have real-time information") or guessed wrong. This is a **structural limitation** — not a model quality issue. The fix is live retrieval, not a bigger model.

---

# Part 2 — Live Search with DuckDuckGo

[`ddgs`](https://pypi.org/project/ddgs/) wraps DuckDuckGo search — free, no API key, no account. One call returns live web snippets we can inject into the prompt.

**Two-step pattern:**
1. LLM distils the question into a short search phrase (how humans actually search)
2. `web_search()` fetches live results → inject into prompt → LLM gives a current, cited answer

In [4]:
from ddgs import DDGS
from datetime import datetime


def make_search_query(question: str) -> str:
    """
    Distil a natural-language question into a concise search phrase (4-6 words).

    Sending a full sentence to a search engine returns noisy results and may
    trigger rate-limiting. A short phrase is how people actually use search.

    Example:
      question → "Who won Super Bowl LXI in 2026?"
      phrase   → "Super Bowl LXI 2026 winner"
    """
    prompt = (
        "Convert the following question into a short web search query (4-6 words). "
        "Output ONLY the search phrase — no explanation, no punctuation.\n\n"
        f"Question: {question}\n"
        "Search phrase:"
    )
    raw    = generate(prompt, max_new_tokens=15).strip()
    phrase = raw.splitlines()[0].strip().strip('"').strip("'")
    print(f"  Question : {question}")
    print(f"  → Query  : {phrase}")
    return phrase


def web_search(query: str, top_k: int = 5) -> list:
    """
    Search the live web via DuckDuckGo. No API key. No account.

    Returns a list of dicts with keys: title, url, snippet.
    """
    try:
        raw = list(DDGS().text(query, max_results=top_k))
        results = [
            {
                "title":   r.get("title", "").strip(),
                "url":     r.get("href", ""),
                "snippet": r.get("body", "").strip(),
                "engine":  "DuckDuckGo",
            }
            for r in raw
        ]
        print(f"  [DDG] {len(results)} results ✓")
        return results
    except Exception as e:
        print(f"  [DDG] Search failed: {type(e).__name__}: {e}")
        return []


def print_results(results: list) -> None:
    if not results:
        print("No results.")
        return
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] {r['title']}")
        print(f"    {r['url']}")
        print(f"    {r['snippet'][:200]}...")

In [5]:
# ── 2.1  Run a live search ───────────────────────────────────────────────────
# Step 1: LLM distils the question into a short search phrase.
# Step 2: DuckDuckGo fetches live results for that phrase.

print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# We bypass make_search_query here because small models often misread Roman numerals
# (e.g. LXI → XLIX). For the demo we use a known-good phrase instead.
search_phrase = make_search_query(question)
# Override if the model garbled the Roman numeral
if 'lxi' not in search_phrase.lower() and 'super bowl 61' not in search_phrase.lower():
    search_phrase = "Super Bowl 61 2026 winner"
    print(f"  (phrase overridden to: \"{search_phrase}\")")
print()

results = web_search(search_phrase, top_k=5)
print_results(results)

Timestamp: 2026-03-21 19:31:26

  Question : Who won Super Bowl LXI in 2026?
  → Query  : Who won Super Bowl XLIX in 2026?
  (phrase overridden to: "Super Bowl 61 2026 winner")

  [DDG] 5 results ✓

[1] Super Bowl 2027 odds: Here’s the best bet to win Super Bowl
    https://www.nj.com/sports/2026/02/super-bowl-2027-odds-heres-the-best-bet-to-win-super-bowl-61.html
    In odds posted by FanDuel , the Seattle Seahawks, thewinnersofSuperBowl60, are the favorites to win next year (+750)....

[2] Super Bowl 2026 Squares Winners: Live Updates and Results (2026)
    https://reikitalia.com/article/super-bowl-2026-squares-winners-live-updates-and-results
    Get ready for a night of heart-pounding excitement as the New England Patriots and Seattle Seahawks clash inSuperBowl2026! But here's the real ......

[3] Seahawks win 2026 Super Bowl. Here are the highlights from the
    https://www.cbsnews.com/live-updates/super-bowl-2026-seattle-seahawks-new-england-patriots/
    MOB!SUPERBOWLLX2026CHAMP

In [6]:
# ── Helper: format results into a grounded prompt ───────────────────────────
def format_as_context(results: list) -> str:
    """
    Format search results as a numbered context block for the LLM.
    Each entry includes title, URL, and snippet so the LLM can cite sources.
    """
    if not results:
        return "[No search results available.]"
    lines = []
    for i, r in enumerate(results, 1):
        lines.append(
            f"[Result {i}]\n"
            f"Title: {r['title']}\n"
            f"URL: {r['url']}\n"
            f"Snippet: {r['snippet']}"
        )
    return "\n\n".join(lines)


def grounded_live_prompt(question: str, results: list) -> str:
    """
    Build a grounded QA prompt from live search results.

    The LLM is instructed to:
      1. Answer ONLY from the provided web results.
      2. Cite the result number and URL for every claim.
      3. Say "Not found in provided results" if snippets don't support an answer.
    This enforces grounding — hallucinations should fail the 'cite your source' test.
    """
    context = format_as_context(results)
    return (
        "You are a research assistant. Answer the question using ONLY the web search results below.\n"
        "For every factual claim, cite the result number and URL (e.g. [Result 2] https://...).\n"
        "If the results do not contain enough information, say \"Not found in provided results.\"\n"
        "\n--- WEB SEARCH RESULTS ---\n"
        f"{context}\n"
        "--------------------------\n"
        f"\nQuestion: {question}\nAnswer:"
    )


In [7]:
# ── 2.2  LLM with live search context ───────────────────────────────────────
# We inject the live search results into the prompt and ask the same question.

prompt = grounded_live_prompt(question, results)

print("─" * 60)
print("LLM answer (grounded in live search results):")
print("─" * 60)
live_answer = generate(prompt)
print(live_answer)

────────────────────────────────────────────────────────────
LLM answer (grounded in live search results):
────────────────────────────────────────────────────────────
Seattle Seahawks


### Part 2 — Key takeaway

| | LLM-only | LLM + live search |
|---|---|---|
| Answer currency | Training-cutoff bound | Current as of search time |
| Citations | None | URL per claim |
| Verifiability | Must trust the model | Click the URL |

The LLM is the same model — only the **context** changed.

In [8]:
# ── 2.3  Three more live questions ───────────────────────────────────────────
# These showcase the breadth of what live-corpus handles — topics that change
# constantly and span current events, recent awards, and evolving policy.

# Simple, direct questions — each has one clear, searchable answer.
# The model cannot know any of these since they all postdate its training cutoff.
live_questions = [
    "Did Sean Penn win an Oscar in 2026?",
    "What happened to the Moon on March 3, 2026?",
    "What is blooming in the University of Washington in March 2026?",
]

for i, q in enumerate(live_questions, 1):
    print("=" * 70)
    print(f"Live Question {i}: {q}")
    print("=" * 70)

    res = web_search(q, top_k=4)

    print("\nTop result snippets:")
    for j, r in enumerate(res[:2], 1):   # show only top-2 snippets to keep output readable
        print(f"  [{j}] {r['title']} | {r['url']}")
        print(f"       {r['snippet'][:150]}...")

    answer = generate(grounded_live_prompt(q, res))
    print(f"\nLLM answer:\n{answer}\n")

Live Question 1: Did Sean Penn win an Oscar in 2026?
  [DDG] 4 results ✓

Top result snippets:
  [1] SeanPenn- Wikipedia | https://en.wikipedia.org/wiki/Sean_Penn
       SeanJustinPennwas born on August 17, 1960, in Santa Monica, California,[4] to actor and director LeoPennand actress Eileen Ryan (née Annucci).[4][5] H...
  [2] SeanPennis awarded (a version of) hisOscarinUkraine | CNN | https://edition.cnn.com/2026/03/18/entertainment/sean-penn-ukraine-oscars
       SeanPenngifted makeshift ‘Oscar’ after skipping ceremony for Ukraine.BeforePenn’swinfor Paul Thomas Anderson’s “One Battle” this weekend, he had previ...

LLM answer:
Yes

Live Question 2: What happened to the Moon on March 3, 2026?
  [DDG] 4 results ✓

Top result snippets:
  [1] March2026lunar eclipse - Wikipedia | https://en.wikipedia.org/wiki/March_2026_lunar_eclipse
       A total lunar eclipse occurred attheMoon’s descending node of orbit on Tuesday,March3,2026, with an umbral magnitude of 1.1507....
  [2] WormMoon2026

---

# Recap — Complexity Ladder

| Level | Name | Source | Freshness | Key challenge |
|-------|------|--------|-----------|---------------|
| 2 | Structured Lookup | API | Provider-managed | Schema design |
| 3 | Grounded Closed-Corpus | Fixed corpus (RAG) | Index time | Retrieval quality |
| **4** | **Grounded Live-Corpus** | **Open web** | **Always current** | **Source credibility + faithfulness** |
| 5 | Agentic Navigation | Multi-step tool use | Always current | Planning + loop termination |

**Level 4 requires three evaluation checks:**
- **Freshness** — is the source recent enough?
- **Credibility** — is the source trustworthy?
- **Faithfulness** — does the LLM answer match the retrieved text?